In [ ]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 107.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
!pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


## Local Inference on GPU
Model page: https://huggingface.co/casperhansen/llama-3-8b-instruct-awq

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/casperhansen/llama-3-8b-instruct-awq)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
!pip install gptqmodel

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 979.1/979.1 kB 22.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.8/121.8 kB 13.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.8/97.8 kB 10.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  

In [ ]:
!uv pip install vllm==0.18.0 --torch-backend=auto

Using Python 3.12.13 environment at: /usr
Resolved 174 packages in 4.35s
Prepared 75 packages in 1m 21s
Uninstalled 14 packages in 981ms
Installed 75 packages in 667ms
 + anthropic==0.105.2
 + apache-tvm-ffi==0.1.11
 + astor==0.8.1
 + blake3==1.0.8
 + cbor2==6.1.1
 + compressed-tensors==0.13.0
 - cuda-bindings==12.9.6
 + cuda-bindings==13.0.3
 - cuda-python==12.9.6
 + cuda-python==13.0.3
 + depyf==0.20.0
 + detect-installer==0.1.0
 + diskcache==5.6.3
 + dnspython==2.8.0
 + email-validator==2.3.0
 + fastapi-cli==0.0.24
 + fastapi-cloud-cli==0.18.0
 + fastar==0.11.0
 + flashinfer-python==0.6.6
 + gguf==0.19.0
 - huggingface-hub==1.16.1
 + huggingface-hub==0.36.2
 + ijson==3.5.0
 + interegular==0.3.3
 + jmespath==1.1.0
 - lark==1.3.1
 + lark==1.2.2
 + llguidance==1.3.0
 - llvmlite==0.43.0
 + llvmlite==0.44.0
 + lm-format-enforcer==0.11.3
 + loguru==0.7.3
 + mistral-common==1.11.2
 + model-hosting-container-standards==0.1.15
 + msgspec==0.21.1
 - numba==0.60.0
 + numba==0.61.2
 + nvidia-cu

In [ ]:
import os
import sys
import contextlib
# Required for runtime swap
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

# Import vLLM's system utils explicitly
import vllm.utils.system_utils

# Create a dummy context manager that skips the fileno() logic
@contextlib.contextmanager
def colab_safe_suppress():
    yield  # Do absolutely nothing

# Neutralize vLLM's native suppress_stdout with our dummy function
vllm.utils.system_utils.suppress_stdout = colab_safe_suppress

In [ ]:
# Library imports
from vllm import LLM, SamplingParams
import torch
import torch.nn.functional as F
import triton
import triton.language as tl
import torch.nn as nn

from vllm.model_executor.models.llama import LlamaMLP

In [ ]:
# Set runtime device
DEVICE = triton.runtime.driver.active.get_active_torch_device()

In [ ]:
@triton.jit
def swiglu(x_ptr, out_ptr, token_stride, hidden_dim_stride, out_stride_x, out_stride_y, num_tokens, hidden_dim, block_size_m: tl.constexpr, block_size_k: tl.constexpr):
  """
  Kernel which applies the SwiGLU activation function to the input matrix.

  Input parameters:
  x_ptr: Pointer to the input matrix.
  out_ptr: Pointer to the output matrix.
  token_stride: Stride between consecutive tokens in the input matrix.
  hidden_dim_stride: Stride between consecutive hidden dimensions in the input matrix.
  out_stride_x: Stride between consecutive tokens in the output matrix.
  out_stride_y: Stride between consecutive hidden dimensions in the output matrix.
  num_tokens: Number of tokens in the input matrix.
  hidden_dim: Number of hidden dimensions in the input matrix.
  block_size_m: Block size for the number of tokens.
  block_size_k: Block size for the number of hidden dimensions.
  """

  # Get 1D block id
  pid = tl.program_id(axis = 0)

  # Thread id with which block starts
  block_start = pid * block_size_m

  # Get the offsets for the number of tokens for a specific block
  token_offsets = block_start + tl.arange(0, block_size_m)

  # Get the offsets for the hidden dimension. Since each block has to apply it on the entire vector, it starts from zero.
  hidden_dim_offsets = tl.arange(0, block_size_k)

  # Mask for handling the final block
  token_mask = token_offsets < num_tokens

  # Calculate the offsets for the gate and up matrices in 2D.
  gate_offsets = token_offsets[:, None] * token_stride + hidden_dim_offsets[None, :] * hidden_dim_stride
  up_offsets = token_offsets[:, None] * token_stride + (hidden_dim + hidden_dim_offsets)[None, :] * hidden_dim_stride

  # Calculate offsets for writing to the output matrix.
  output_offsets = token_offsets[:, None] * out_stride_x + hidden_dim_offsets[None, :] * out_stride_y

  # Loop over the hidden dimension
  for col_group in range(tl.cdiv(hidden_dim, block_size_k)):
    # Mask for handling when offsets go beyond the hidden dimension
    hidden_dim_mask = hidden_dim_offsets < hidden_dim

    # Load the actual values
    gate_mat = tl.load(x_ptr + gate_offsets, mask = token_mask[:, None] & hidden_dim_mask[None, :], other = 0.0)
    up_mat = tl.load(x_ptr + up_offsets, mask = token_mask[:, None] & hidden_dim_mask[None, :], other = 0.0)

    # Upcasting from float16 to float32 to prevent underflow
    gate_mat = gate_mat.to(tl.float32)
    up_mat = up_mat.to(tl.float32)

    # SwiGLU activation
    out_mat_fp32 = gate_mat * tl.sigmoid(gate_mat) * up_mat

    # Downcast output to match model architecture
    out_mat = out_mat_fp32.to(tl.float16)

    # Store it in the output matrix
    tl.store(out_ptr + output_offsets, out_mat, mask = token_mask[:, None] & hidden_dim_mask[None, :])

    # Step over each offset
    hidden_dim_offsets += block_size_k
    gate_offsets += block_size_k * hidden_dim_stride
    up_offsets += block_size_k * hidden_dim_stride
    output_offsets += block_size_k * out_stride_y

In [ ]:
class customSwiglu(nn.Module):
  """Class which calls the kernel."""

  def __init__(self):
    """Init function."""
    super().__init__()

  def forward(self, x):
    """Forward method which invokes the kernel."""
    # Save the original shape for the output matrix
    original_shape = x.shape

    # Collapse the first two dimensions to maintain token level dimensionality.
    squashed_x = x.view([-1, x.shape[-1]])

    # Total number of tokens being passed
    total_tokens = squashed_x.shape[0]

    # Since vLLM passes the gate and up matrices in a single matrix by concatening them side by side, the column width is twice the hidden dimension.
    hidden_dim = squashed_x.shape[1] // 2

    # Initialize the output matrix
    output = torch.empty((total_tokens, hidden_dim), device = x.device, dtype = x.dtype)

    # Design a 1D grid of blocks
    grid = lambda meta: ((triton.cdiv(total_tokens, meta['block_size_m'])),)

    # Invoke the kernel
    swiglu[grid](squashed_x, output, squashed_x.stride(0), squashed_x.stride(1), output.stride(0), output.stride(1), total_tokens, hidden_dim, block_size_m = 64, block_size_k = 32)

    # Get the shape based on the original hidden dimension
    new_shape = original_shape[:-1] + (hidden_dim,)

    # Reshape and return
    return output.view(new_shape)

In [ ]:
# Custom function object
custom_swiglu = customSwiglu()

def patched_llama_forward(self, x):
  """Function definition based on LlamaMLP.forward."""
  # Execute the native AWQ projection
  x, _ = self.gate_up_proj(x)

  # Bypass self.act_fn(x) entirely
  x = custom_swiglu(x)

  # Finish the standard layer execution
  x, _ = self.down_proj(x)
  return x

# Execute the runtime swap
LlamaMLP.forward = patched_llama_forward

In [ ]:
# Initialize the vLLM object
llm  = LLM(model="casperhansen/llama-3-8b-instruct-awq", quantization = "awq", tensor_parallel_size = 1, enforce_eager = True)

INFO 05-29 19:17:49 [utils.py:233] non-default args: {'disable_log_stats': True, 'quantization': 'awq', 'enforce_eager': True, 'model': 'casperhansen/llama-3-8b-instruct-awq'}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/975 [00:00<?, ?B/s]

WARNING 05-29 19:17:50 [arg_utils.py:1352] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 05-29 19:18:11 [model.py:533] Resolved architecture: LlamaForCausalLM
INFO 05-29 19:18:11 [model.py:1582] Using max model len 8192
INFO 05-29 19:18:12 [awq_marlin.py:166] Detected that the model can run with awq_marlin, however you specified quantization=awq explicitly, so forcing awq. Use quantization=awq_marlin for faster inference
INFO 05-29 19:18:12 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.05G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.68G [00:00<?, ?B/s]

INFO 05-29 19:19:26 [weight_utils.py:574] Time spent downloading weights for casperhansen/llama-3-8b-instruct-awq: 67.815480 seconds


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 05-29 19:19:44 [default_loader.py:384] Loading weights took 17.80 seconds
INFO 05-29 19:19:45 [gpu_model_runner.py:4566] Model loading took 5.34 GiB memory and 87.300056 seconds
INFO 05-29 19:20:05 [gpu_worker.py:456] Available KV cache memory: 6.86 GiB
INFO 05-29 19:20:05 [kv_cache_utils.py:1316] GPU KV cache size: 56,224 tokens
INFO 05-29 19:20:05 [kv_cache_utils.py:1321] Maximum concurrency for 8,192 tokens per request: 6.86x
INFO 05-29 19:20:05 [kernel_warmup.py:69] Warming up FlashInfer attention.
INFO 05-29 19:22:40 [core.py:281] init engine (profile, create kv cache, warmup model) took 174.69 seconds
INFO 05-29 19:22:42 [llm.py:391] Supported tasks: ('generate',)


In [ ]:
# Set parameters for the output
params = SamplingParams(temperature=0.8, max_tokens = 100)

# Get the tokenizer for applying chat template
tokenizer = llm.get_tokenizer()

In [ ]:
# Block to fetch prompts from the alpaca dataset
from datasets import load_dataset
ds = load_dataset("tatsu-lab/alpaca", split = "train")
inputs = ds["instruction"][:200]

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

In [ ]:
formatted_prompts = []
for prompt in inputs:
  # Wrap the raw text in the standard dictionary format
  message = [{"role": "user", "content": prompt}]

  # Apply the template to create the raw string Llama-3 expects
  formatted_string = tokenizer.apply_chat_template(
      conversation=message,
      tokenize=False,
      add_generation_prompt=True
  )

  formatted_prompts.append(formatted_string)

In [ ]:
# Output for 50 prompts

# outputs = llm.generate(formatted_prompts, sampling_params = params)

# for output in outputs:
#     prompt = output.prompt
#     generated_text = output.outputs[0].text
#     print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}")

Rendering prompts:   0%|          | 0/50 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/50 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Prompt: "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{'role': 'user', 'content': 'What is the capital of Australia?'}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n", Generated text: 'The capital of Australia is Canberra.'
Prompt: '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nExplain the theory of relativity in exactly one sentence.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n', Generated text: 'According to the theory of relativity, the laws of physics are the same for all observers, regardless of their relative motion, and time and space are relative and can be affected by gravity and motion, leading to concepts such as time dilation and length contraction.'
Prompt: '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nWho won the World Series in 2016?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n', Generated text: 'The Chicago Cubs won the World Series in 2016, ending a 108-year championship drought. The

In [ ]:
outputs = llm.generate(formatted_prompts, sampling_params = params)

for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}")

Rendering prompts:   0%|          | 0/200 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/200 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Prompt: '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nGive three tips for staying healthy.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n', Generated text: 'Here are three tips for staying healthy:\n\n1. **Stay Hydrated**: Drink plenty of water throughout the day to help your body function properly. Aim for at least 8 cups (64 ounces) of water daily. You can also consume other hydrating fluids like herbal tea, low-sugar sports drinks, and water-rich foods like fruits and vegetables.\n2. **Get Moving**: Regular exercise is essential for maintaining physical and mental health. Aim for at least 30 minutes of moderate-intensity'
Prompt: '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nWhat are the three primary colors?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n', Generated text: 'The three primary colors are:\n\n1. **Red**\n2. **Blue**\n3. **Yellow**\n\nThese three colors cannot be created by mixing other colors together, and they ar